# 銘柄スイング適性分析ツール
**銘柄ごとに有効なサインを発見する**

- 直近2年：銘柄の現在の特性把握
- 過去10年：各サインの統計的有効性検証
- Claude AIによる総合分析・推奨サイン特定

In [ ]:
# ===== セル1：インストール =====
!pip install -q yfinance pandas numpy matplotlib anthropic

In [ ]:
# ===== セル2：設定 =====
# ここを編集して実行

ANTHROPIC_API_KEY = "sk-ant-XXXXXXXX"  # ← APIキーを入力

# 分析したい銘柄コード（東証は末尾に.T）
# 複数入れると全部分析します
TICKERS = [
    "7203.T",  # トヨタ
    # "6758.T",  # ソニーG
    # "8316.T",  # 三井住友
]

print(f"分析銘柄: {TICKERS}")
print("設定完了。次のセルを実行してください。")

In [ ]:
# ===== セル3：データ取得・指標計算 =====
import yfinance as yf
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

def calc_indicators(df):
    """全指標を計算して返す"""
    d = df.copy()
    close = d['Close']
    high  = d['High']
    low   = d['Low']
    vol   = d['Volume']

    # --- 移動平均 ---
    for n in [5, 25, 50, 75, 200]:
        d[f'SMA{n}'] = close.rolling(n).mean()
    for n in [5, 20, 25]:
        d[f'EMA{n}'] = close.ewm(span=n, adjust=False).mean()

    # --- SMAの傾き（5日前比） ---
    d['SMA50_slope'] = d['SMA50'] - d['SMA50'].shift(5)
    d['SMA25_slope'] = d['SMA25'] - d['SMA25'].shift(5)

    # --- RSI ---
    for n in [2, 9, 14]:
        delta = close.diff()
        gain  = delta.clip(lower=0).rolling(n).mean()
        loss  = (-delta.clip(upper=0)).rolling(n).mean()
        rs    = gain / loss.replace(0, np.nan)
        d[f'RSI{n}'] = 100 - (100 / (1 + rs))

    # --- ボリンジャーバンド ---
    bb_mid = close.rolling(20).mean()
    bb_std = close.rolling(20).std()
    d['BB_upper'] = bb_mid + 2 * bb_std
    d['BB_lower'] = bb_mid - 2 * bb_std
    d['BB_mid']   = bb_mid
    d['BB_width'] = (d['BB_upper'] - d['BB_lower']) / bb_mid  # バンド幅%
    d['BB_pct_b'] = (close - d['BB_lower']) / (d['BB_upper'] - d['BB_lower'])

    # --- スクイーズ判定（BB幅が過去30日の下位20%） ---
    d['BB_squeeze'] = d['BB_width'] <= d['BB_width'].rolling(30).quantile(0.20)

    # --- ATR ---
    tr = pd.concat([
        high - low,
        (high - close.shift()).abs(),
        (low  - close.shift()).abs()
    ], axis=1).max(axis=1)
    d['ATR14'] = tr.rolling(14).mean()
    d['ATR_pct'] = d['ATR14'] / close * 100  # %ATR

    # --- MACD ---
    ema12 = close.ewm(span=12, adjust=False).mean()
    ema26 = close.ewm(span=26, adjust=False).mean()
    d['MACD']        = ema12 - ema26
    d['MACD_signal'] = d['MACD'].ewm(span=9, adjust=False).mean()
    d['MACD_hist']   = d['MACD'] - d['MACD_signal']

    # --- ADX ---
    plus_dm  = high.diff().clip(lower=0)
    minus_dm = (-low.diff()).clip(lower=0)
    plus_dm  = plus_dm.where(plus_dm > minus_dm, 0)
    minus_dm = minus_dm.where(minus_dm > plus_dm, 0)
    atr14    = tr.ewm(alpha=1/14, adjust=False).mean()
    plus_di  = 100 * plus_dm.ewm(alpha=1/14, adjust=False).mean() / atr14
    minus_di = 100 * minus_dm.ewm(alpha=1/14, adjust=False).mean() / atr14
    dx       = (100 * (plus_di - minus_di).abs() / (plus_di + minus_di).replace(0, np.nan))
    d['ADX']      = dx.ewm(alpha=1/14, adjust=False).mean()
    d['ADX_slope'] = d['ADX'] - d['ADX'].shift(5)

    # --- 出来高 ---
    d['Vol_SMA5']  = vol.rolling(5).mean()
    d['Vol_SMA20'] = vol.rolling(20).mean()
    d['Vol_ratio'] = vol / d['Vol_SMA20']  # 出来高比率

    # --- 安値の切り上がり（直近3安値） ---
    lows = low.rolling(5).min()
    d['Low_rising'] = (lows > lows.shift(5)) & (lows.shift(5) > lows.shift(10))

    # --- GC/DC判定 ---
    d['GC_5_25']  = (d['SMA5']  > d['SMA25'])  & (d['SMA5'].shift()  <= d['SMA25'].shift())
    d['GC_25_75'] = (d['SMA25'] > d['SMA75'])  & (d['SMA25'].shift() <= d['SMA75'].shift())
    d['DC_5_25']  = (d['SMA5']  < d['SMA25'])  & (d['SMA5'].shift()  >= d['SMA25'].shift())

    return d


def get_stock_data(ticker):
    """2年分・10年分を取得"""
    print(f"  {ticker} データ取得中...")
    df_10y = yf.download(ticker, period="10y", interval="1d", auto_adjust=True, progress=False)
    df_2y  = yf.download(ticker, period="2y",  interval="1d", auto_adjust=True, progress=False)

    if isinstance(df_10y.columns, pd.MultiIndex):
        df_10y.columns = df_10y.columns.get_level_values(0)
    if isinstance(df_2y.columns, pd.MultiIndex):
        df_2y.columns = df_2y.columns.get_level_values(0)

    df_10y = calc_indicators(df_10y.dropna())
    df_2y  = calc_indicators(df_2y.dropna())
    return df_10y, df_2y


# データ取得実行
all_data = {}
for ticker in TICKERS:
    try:
        df_10y, df_2y = get_stock_data(ticker)
        all_data[ticker] = {'10y': df_10y, '2y': df_2y}
        print(f"  ✓ {ticker}: 10年={len(df_10y)}日, 2年={len(df_2y)}日")
    except Exception as e:
        print(f"  ✗ {ticker}: エラー - {e}")

print("\nデータ取得完了。次のセルを実行してください。")

In [ ]:
# ===== セル4：サイン検証エンジン =====

def simulate_trade(df, entry_idx, direction='long', atr_k_sl=1.5, atr_k_tp=3.0, max_hold=10):
    """
    エントリーから最大max_hold日のトレードをシミュレート
    direction: 'long' or 'short'
    returns: dict(result, hold_days, ret, rr, hit_tp, hit_sl)
    """
    if entry_idx + max_hold >= len(df):
        return None

    entry_price = df['Close'].iloc[entry_idx + 1]  # 翌日寄り近似
    atr = df['ATR14'].iloc[entry_idx]
    if pd.isna(atr) or atr <= 0:
        return None

    sl_dist = atr * atr_k_sl
    tp_dist = atr * atr_k_tp

    if direction == 'long':
        sl_price = entry_price - sl_dist
        tp_price = entry_price + tp_dist
    else:
        sl_price = entry_price + sl_dist
        tp_price = entry_price - tp_dist

    for i in range(1, max_hold + 1):
        idx = entry_idx + 1 + i
        if idx >= len(df):
            break
        h = df['High'].iloc[idx]
        l = df['Low'].iloc[idx]
        c = df['Close'].iloc[idx]

        if direction == 'long':
            if l <= sl_price:
                ret = (sl_price - entry_price) / entry_price
                return dict(result='sl', hold_days=i, ret=ret, rr=-atr_k_sl, hit_tp=False, hit_sl=True)
            if h >= tp_price:
                ret = (tp_price - entry_price) / entry_price
                return dict(result='tp', hold_days=i, ret=ret, rr=atr_k_tp, hit_tp=True, hit_sl=False)
        else:
            if h >= sl_price:
                ret = (entry_price - sl_price) / entry_price
                return dict(result='sl', hold_days=i, ret=-ret, rr=-atr_k_sl, hit_tp=False, hit_sl=True)
            if l <= tp_price:
                ret = (entry_price - tp_price) / entry_price
                return dict(result='tp', hold_days=i, ret=ret, rr=atr_k_tp, hit_tp=True, hit_sl=False)

    # 期間満了
    exit_price = df['Close'].iloc[min(entry_idx + 1 + max_hold, len(df)-1)]
    if direction == 'long':
        ret = (exit_price - entry_price) / entry_price
    else:
        ret = (entry_price - exit_price) / entry_price
    return dict(result='timeout', hold_days=max_hold, ret=ret, rr=ret/((atr*atr_k_sl)/entry_price), hit_tp=False, hit_sl=False)


def test_sign(df, signal_mask, direction='long', label='', min_samples=10):
    """
    シグナルマスクに対してトレードシミュレーションを実行
    returns: dict with stats
    """
    indices = np.where(signal_mask)[0]
    trades = []

    for idx in indices:
        t = simulate_trade(df, int(idx))
        if t:
            trades.append(t)

    if len(trades) < min_samples:
        return None

    rets  = np.array([t['ret'] for t in trades])
    wins  = rets[rets > 0]
    loses = rets[rets < 0]
    tp_count = sum(1 for t in trades if t['hit_tp'])
    hold_days = np.mean([t['hold_days'] for t in trades])

    win_rate = len(wins) / len(trades)
    avg_win  = wins.mean()  if len(wins)  > 0 else 0
    avg_loss = loses.mean() if len(loses) > 0 else 0
    pf = (wins.sum() / abs(loses.sum())) if len(loses) > 0 and loses.sum() != 0 else float('inf')
    expectancy = win_rate * avg_win + (1 - win_rate) * avg_loss
    # 年間頻度（取引日数で計算）
    years = len(df) / 252
    freq_per_year = len(trades) / years if years > 0 else 0

    return dict(
        label=label,
        direction=direction,
        samples=len(trades),
        win_rate=round(win_rate * 100, 1),
        tp_rate=round(tp_count / len(trades) * 100, 1),
        avg_ret=round(float(rets.mean()) * 100, 2),
        avg_win=round(float(avg_win) * 100, 2),
        avg_loss=round(float(avg_loss) * 100, 2),
        pf=round(float(pf), 2),
        expectancy=round(float(expectancy) * 100, 3),
        avg_hold_days=round(float(hold_days), 1),
        freq_per_year=round(freq_per_year, 1),
    )


def analyze_signs(df_10y, df_2y):
    """
    全サインを検証して結果を返す
    """
    results = []
    d = df_10y

    signs = [
        # ---- 単体サイン ----
        {
            'label': 'BBスクイーズ→上方ブレイク（単体）',
            'mask': d['BB_squeeze'].shift(1).fillna(False) & (d['Close'] > d['BB_upper']),
            'dir': 'long'
        },
        {
            'label': 'GC(5/25)単体',
            'mask': d['GC_5_25'],
            'dir': 'long'
        },
        {
            'label': 'GC(25/75)単体',
            'mask': d['GC_25_75'],
            'dir': 'long'
        },
        {
            'label': 'RSI(2)売られすぎ回復（<15から50超え）',
            'mask': (d['RSI2'].shift(1) < 15) & (d['RSI2'] > 50),
            'dir': 'long'
        },
        {
            'label': 'RSI(14)売られすぎ回復（<30から50超え）',
            'mask': (d['RSI14'].shift(1) < 30) & (d['RSI14'] > 50),
            'dir': 'long'
        },
        {
            'label': 'MACD GC（ゼロライン下）',
            'mask': (d['MACD'] > d['MACD_signal']) & (d['MACD'].shift() <= d['MACD_signal'].shift()) & (d['MACD'] < 0),
            'dir': 'long'
        },
        {
            'label': 'MACD GC（ゼロライン上）',
            'mask': (d['MACD'] > d['MACD_signal']) & (d['MACD'].shift() <= d['MACD_signal'].shift()) & (d['MACD'] > 0),
            'dir': 'long'
        },
        {
            'label': '出来高急増×高値更新',
            'mask': (d['Vol_ratio'] > 2.0) & (d['Close'] > d['Close'].rolling(20).max().shift(1)),
            'dir': 'long'
        },

        # ---- 組み合わせサイン ----
        {
            'label': 'BBスクイーズ→上方ブレイク × SMA50上向き',
            'mask': (
                d['BB_squeeze'].shift(1).fillna(False)
                & (d['Close'] > d['BB_upper'])
                & (d['SMA50_slope'] > 0)
            ),
            'dir': 'long'
        },
        {
            'label': 'BBスクイーズ→上方ブレイク × ADX上昇',
            'mask': (
                d['BB_squeeze'].shift(1).fillna(False)
                & (d['Close'] > d['BB_upper'])
                & (d['ADX'] > 20)
                & (d['ADX_slope'] > 0)
            ),
            'dir': 'long'
        },
        {
            'label': 'BBスクイーズ → GC(5/25) 同時',
            'mask': (
                d['BB_squeeze'].shift(1).fillna(False)
                & (d['Close'] > d['BB_upper'])
                & d['GC_5_25']
            ),
            'dir': 'long'
        },
        {
            'label': 'GC(5/25) × SMA200上 × 出来高増加',
            'mask': (
                d['GC_5_25']
                & (d['Close'] > d['SMA200'])
                & (d['Vol_ratio'] > 1.5)
            ),
            'dir': 'long'
        },
        {
            'label': 'RSI(2)売られすぎ × SMA200上 × 安値切り上がり',
            'mask': (
                (d['RSI2'] < 15)
                & (d['Close'] > d['SMA200'])
                & d['Low_rising']
            ),
            'dir': 'long'
        },
        {
            'label': 'MACD GC × 出来高急増',
            'mask': (
                (d['MACD'] > d['MACD_signal'])
                & (d['MACD'].shift() <= d['MACD_signal'].shift())
                & (d['Vol_ratio'] > 1.5)
            ),
            'dir': 'long'
        },
        {
            'label': '押し目スクイーズ（SMA50上×RSI回復×BB収縮→拡大）',
            'mask': (
                d['BB_squeeze'].shift(1).fillna(False)
                & (d['Close'] > d['BB_upper'])
                & (d['SMA50_slope'] > 0)
                & (d['RSI14'].shift(3) < 40)
                & (d['RSI14'] > 50)
            ),
            'dir': 'long'
        },
    ]

    for s in signs:
        try:
            mask = s['mask'].fillna(False).values
            r = test_sign(d, mask, direction=s['dir'], label=s['label'])
            if r:
                results.append(r)
        except Exception as e:
            print(f"  スキップ {s['label']}: {e}")

    # 直近2年の特性
    d2 = df_2y.dropna()
    recent_stats = dict(
        avg_atr_pct=round(float(d2['ATR_pct'].mean()), 2),
        avg_adx=round(float(d2['ADX'].mean()), 1),
        bb_squeeze_days=int(d2['BB_squeeze'].sum()),
        vol_ratio_mean=round(float(d2['Vol_ratio'].mean()), 2),
        close_vs_sma200=bool((d2['Close'].iloc[-1] > d2['SMA200'].iloc[-1])),
        sma50_slope_recent=round(float(d2['SMA50_slope'].iloc[-1]), 2),
        latest_rsi14=round(float(d2['RSI14'].iloc[-1]), 1),
        latest_adx=round(float(d2['ADX'].iloc[-1]), 1),
        latest_bb_width=round(float(d2['BB_width'].iloc[-1]) * 100, 2),
        latest_vol_ratio=round(float(d2['Vol_ratio'].iloc[-1]), 2),
        latest_close=round(float(d2['Close'].iloc[-1]), 1),
    )

    return results, recent_stats


# 検証実行
sign_results = {}
recent_stats_all = {}

for ticker, data in all_data.items():
    print(f"\n{ticker} サイン検証中...")
    results, recent_stats = analyze_signs(data['10y'], data['2y'])
    sign_results[ticker] = sorted(results, key=lambda x: x['win_rate'], reverse=True)
    recent_stats_all[ticker] = recent_stats
    print(f"  有効サイン数: {len(results)}")

print("\nサイン検証完了。次のセルを実行してください。")

In [ ]:
# ===== セル5：Claude分析 + HTML表示 =====
import anthropic
import json
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import base64
from io import BytesIO
from IPython.display import display, HTML

matplotlib.rcParams['font.family'] = 'IPAexGothic'

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)


def chart_to_base64(ticker, df_2y):
    """直近2年チャート（価格+BB+出来高）をbase64で返す"""
    d = df_2y.dropna().tail(504)  # 2年分
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7),
                                    gridspec_kw={'height_ratios': [3, 1]},
                                    facecolor='#0d1117')
    fig.patch.set_facecolor('#0d1117')

    for ax in [ax1, ax2]:
        ax.set_facecolor('#0d1117')
        ax.tick_params(colors='#8b949e')
        ax.spines['bottom'].set_color('#30363d')
        ax.spines['left'].set_color('#30363d')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

    dates = d.index
    ax1.plot(dates, d['Close'],   color='#58a6ff', linewidth=1.2, label='Close')
    ax1.plot(dates, d['BB_upper'],color='#f78166', linewidth=0.7, linestyle='--', alpha=0.7, label='BB Upper')
    ax1.plot(dates, d['BB_lower'],color='#3fb950', linewidth=0.7, linestyle='--', alpha=0.7, label='BB Lower')
    ax1.plot(dates, d['BB_mid'],  color='#8b949e', linewidth=0.5, linestyle=':', alpha=0.5)
    ax1.plot(dates, d['SMA50'],   color='#ffa657', linewidth=0.8, alpha=0.8, label='SMA50')
    ax1.fill_between(dates, d['BB_upper'], d['BB_lower'], alpha=0.05, color='#58a6ff')

    # スクイーズ期間をハイライト
    squeeze = d['BB_squeeze'].fillna(False)
    ax1.fill_between(dates, d['BB_upper'], d['BB_lower'],
                     where=squeeze, alpha=0.15, color='#f78166', label='Squeeze')

    ax1.set_title(f'{ticker} - 直近2年チャート', color='#e6edf3', fontsize=12, pad=10)
    ax1.legend(loc='upper left', fontsize=8, facecolor='#161b22', edgecolor='#30363d',
               labelcolor='#8b949e')
    ax1.yaxis.label.set_color('#8b949e')

    # 出来高
    colors = ['#3fb950' if c >= o else '#f78166'
              for c, o in zip(d['Close'], d['Open'])]
    ax2.bar(dates, d['Volume'], color=colors, alpha=0.6, width=1)
    ax2.plot(dates, d['Vol_SMA20'], color='#ffa657', linewidth=0.8)
    ax2.set_ylabel('Volume', color='#8b949e', fontsize=8)
    ax2.xaxis.set_major_formatter(mdates.DateFormatter('%y/%m'))

    plt.tight_layout()
    buf = BytesIO()
    plt.savefig(buf, format='png', dpi=100, bbox_inches='tight',
                facecolor='#0d1117')
    plt.close()
    buf.seek(0)
    return base64.b64encode(buf.read()).decode('utf-8')


def ask_claude(ticker, sign_results, recent_stats):
    """Claudeにサイン分析を依頼"""
    top_signs = sign_results[:10] if len(sign_results) >= 10 else sign_results

    prompt = (
        "あなたはスイングトレードの専門家です。以下のデータを分析してください。\n\n"
        f"【銘柄】{ticker}\n"
        f"【直近2年の特性】\n{json.dumps(recent_stats, ensure_ascii=False, indent=2)}\n\n"
        f"【サイン検証結果（上位）】\n{json.dumps(top_signs, ensure_ascii=False, indent=2)}\n\n"
        "以下の観点で分析してください：\n"
        "1. この銘柄のスイング適性（★1〜5）と理由\n"
        "2. 最も有効なサインTop3とその根拠\n"
        "3. 推奨エントリー条件（具体的に）\n"
        "4. 推奨保有期間と出口戦略\n"
        "5. この銘柄特有の注意点・癖\n"
        "6. 総合評価コメント（3行以内）\n\n"
        "評価軸：サインが出てから素直にトレンドが出て利確タイミングがちゃんとあること。"
        "win_rateよりもtp_rate（利確到達率）とpf（プロフィットファクター）を重視してください。\n"
        "必ずJSON形式で返してください：\n"
        "{\"swing_score\": 3, \"top_signs\": [...], "
        "\"entry_condition\": \"...\", \"hold_period\": \"...\", "
        "\"exit_strategy\": \"...\", \"caution\": \"...\", \"summary\": \"...\"}"
    )

    response = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=1500,
        messages=[{"role": "user", "content": prompt}]
    )

    text = response.content[0].text
    # JSON部分を抽出
    start = text.find('{')
    end   = text.rfind('}') + 1
    if start >= 0 and end > start:
        return json.loads(text[start:end])
    return {"summary": text, "swing_score": 0, "top_signs": [], "entry_condition": "",
            "hold_period": "", "exit_strategy": "", "caution": ""}


def render_html(ticker, claude_result, sign_results, recent_stats, chart_b64):
    """分析結果をHTML表示"""
    score = claude_result.get('swing_score', 0)
    stars = '★' * score + '☆' * (5 - score)

    # サイン結果テーブル
    rows = ''
    for r in sign_results[:12]:
        pf_color = '#3fb950' if r['pf'] >= 1.5 else ('#ffa657' if r['pf'] >= 1.0 else '#f78166')
        wr_color = '#3fb950' if r['win_rate'] >= 60 else ('#ffa657' if r['win_rate'] >= 50 else '#f78166')
        rows += (
            "<tr>"
            "<td style='padding:6px 10px;font-size:12px;color:#e6edf3;border-bottom:1px solid #21262d'>" + r['label'] + "</td>"
            "<td style='padding:6px 10px;text-align:center;font-size:12px;border-bottom:1px solid #21262d;color:" + wr_color + "'>" + str(r['win_rate']) + "%</td>"
            "<td style='padding:6px 10px;text-align:center;font-size:12px;border-bottom:1px solid #21262d;color:#58a6ff'>" + str(r['tp_rate']) + "%</td>"
            "<td style='padding:6px 10px;text-align:center;font-size:12px;border-bottom:1px solid #21262d;color:" + pf_color + "'>" + str(r['pf']) + "</td>"
            "<td style='padding:6px 10px;text-align:center;font-size:12px;border-bottom:1px solid #21262d;color:#8b949e'>" + str(r['avg_hold_days']) + "日</td>"
            "<td style='padding:6px 10px;text-align:center;font-size:12px;border-bottom:1px solid #21262d;color:#8b949e'>" + str(r['freq_per_year']) + "回/年</td>"
            "<td style='padding:6px 10px;text-align:center;font-size:12px;border-bottom:1px solid #21262d;color:#8b949e'>" + str(r['samples']) + "</td>"
            "</tr>"
        )

    top_signs_html = ''
    for i, s in enumerate(claude_result.get('top_signs', [])[:3], 1):
        top_signs_html += (
            "<div style='background:#161b22;border:1px solid #30363d;border-radius:8px;"
            "padding:12px;margin-bottom:8px'>"
            "<span style='color:#ffa657;font-weight:bold;font-size:13px'>#" + str(i) + " " + str(s) + "</span>"
            "</div>"
        )

    recent_close_color = '#3fb950' if recent_stats['close_vs_sma200'] else '#f78166'
    sma50_color = '#3fb950' if recent_stats['sma50_slope_recent'] > 0 else '#f78166'

    html = (
        "<div style='font-family:monospace;background:#0d1117;color:#e6edf3;"
        "padding:20px;border-radius:12px;max-width:1000px'>"

        # ヘッダー
        "<div style='display:flex;justify-content:space-between;align-items:center;"
        "border-bottom:2px solid #30363d;padding-bottom:16px;margin-bottom:20px'>"
        "<div>"
        "<h2 style='margin:0;color:#58a6ff;font-size:22px'>" + ticker + "</h2>"
        "<p style='margin:4px 0 0;color:#8b949e;font-size:12px'>銘柄スイング適性分析</p>"
        "</div>"
        "<div style='text-align:right'>"
        "<div style='font-size:28px;color:#ffa657'>" + stars + "</div>"
        "<div style='color:#8b949e;font-size:11px'>スイング適性スコア</div>"
        "</div>"
        "</div>"

        # チャート
        "<img src='data:image/png;base64," + chart_b64 + "' "
        "style='width:100%;border-radius:8px;margin-bottom:20px'>"

        # 直近特性
        "<div style='display:grid;grid-template-columns:repeat(4,1fr);gap:10px;margin-bottom:20px'>"
        "<div style='background:#161b22;border:1px solid #30363d;border-radius:8px;padding:12px;text-align:center'>"
        "<div style='color:#8b949e;font-size:11px'>%ATR（直近2年）</div>"
        "<div style='color:#58a6ff;font-size:18px;font-weight:bold'>" + str(recent_stats['avg_atr_pct']) + "%</div>"
        "</div>"
        "<div style='background:#161b22;border:1px solid #30363d;border-radius:8px;padding:12px;text-align:center'>"
        "<div style='color:#8b949e;font-size:11px'>SMA200 位置</div>"
        "<div style='color:" + recent_close_color + ";font-size:18px;font-weight:bold'>" + ("上" if recent_stats['close_vs_sma200'] else "下") + "</div>"
        "</div>"
        "<div style='background:#161b22;border:1px solid #30363d;border-radius:8px;padding:12px;text-align:center'>"
        "<div style='color:#8b949e;font-size:11px'>SMA50 傾き</div>"
        "<div style='color:" + sma50_color + ";font-size:18px;font-weight:bold'>" + ("↑" if recent_stats['sma50_slope_recent'] > 0 else "↓") + "</div>"
        "</div>"
        "<div style='background:#161b22;border:1px solid #30363d;border-radius:8px;padding:12px;text-align:center'>"
        "<div style='color:#8b949e;font-size:11px'>RSI(14) 現在</div>"
        "<div style='color:#e6edf3;font-size:18px;font-weight:bold'>" + str(recent_stats['latest_rsi14']) + "</div>"
        "</div>"
        "</div>"

        # Claude分析
        "<div style='display:grid;grid-template-columns:1fr 1fr;gap:16px;margin-bottom:20px'>"
        "<div style='background:#161b22;border:1px solid #30363d;border-radius:8px;padding:16px'>"
        "<h3 style='color:#ffa657;margin:0 0 12px;font-size:13px'>▶ 推奨エントリー条件</h3>"
        "<p style='color:#e6edf3;font-size:12px;margin:0;line-height:1.6'>" + str(claude_result.get('entry_condition', '')) + "</p>"
        "</div>"
        "<div style='background:#161b22;border:1px solid #30363d;border-radius:8px;padding:16px'>"
        "<h3 style='color:#3fb950;margin:0 0 12px;font-size:13px'>▶ 出口戦略</h3>"
        "<p style='color:#e6edf3;font-size:12px;margin:0;line-height:1.6'>"
        "<b style='color:#8b949e'>期間：</b>" + str(claude_result.get('hold_period', '')) + "<br>"
        "<b style='color:#8b949e'>利確：</b>" + str(claude_result.get('exit_strategy', '')) + "</p>"
        "</div>"
        "</div>"

        # Top3サイン
        "<div style='margin-bottom:20px'>"
        "<h3 style='color:#58a6ff;font-size:13px;margin-bottom:10px'>▶ 有効サイン Top3</h3>"
        + top_signs_html +
        "</div>"

        # 注意点
        "<div style='background:#1c1007;border:1px solid #3d2b0a;border-radius:8px;"
        "padding:12px;margin-bottom:20px'>"
        "<h3 style='color:#ffa657;margin:0 0 8px;font-size:13px'>⚠ 注意点・銘柄の癖</h3>"
        "<p style='color:#e6edf3;font-size:12px;margin:0'>" + str(claude_result.get('caution', '')) + "</p>"
        "</div>"

        # 総評
        "<div style='background:#0d2137;border:1px solid #0d419d;border-radius:8px;"
        "padding:12px;margin-bottom:24px'>"
        "<h3 style='color:#58a6ff;margin:0 0 8px;font-size:13px'>💡 総合評価</h3>"
        "<p style='color:#e6edf3;font-size:12px;margin:0;line-height:1.6'>" + str(claude_result.get('summary', '')) + "</p>"
        "</div>"

        # サイン検証テーブル
        "<h3 style='color:#8b949e;font-size:13px;margin-bottom:10px'>📊 全サイン検証結果（10年バックテスト）</h3>"
        "<div style='overflow-x:auto'>"
        "<table style='width:100%;border-collapse:collapse;font-size:12px'>"
        "<thead><tr style='background:#161b22'>"
        "<th style='padding:8px 10px;text-align:left;color:#8b949e;border-bottom:2px solid #30363d'>サイン</th>"
        "<th style='padding:8px 10px;color:#8b949e;border-bottom:2px solid #30363d'>勝率</th>"
        "<th style='padding:8px 10px;color:#58a6ff;border-bottom:2px solid #30363d'>利確到達率</th>"
        "<th style='padding:8px 10px;color:#8b949e;border-bottom:2px solid #30363d'>PF</th>"
        "<th style='padding:8px 10px;color:#8b949e;border-bottom:2px solid #30363d'>平均保有</th>"
        "<th style='padding:8px 10px;color:#8b949e;border-bottom:2px solid #30363d'>頻度</th>"
        "<th style='padding:8px 10px;color:#8b949e;border-bottom:2px solid #30363d'>N数</th>"
        "</tr></thead>"
        "<tbody>" + rows + "</tbody>"
        "</table></div></div>"
    )
    display(HTML(html))


# 全銘柄を分析・表示
for ticker in TICKERS:
    if ticker not in all_data:
        continue
    print(f"\n{ticker} Claude分析中...")
    try:
        chart_b64     = chart_to_base64(ticker, all_data[ticker]['2y'])
        claude_result = ask_claude(ticker, sign_results[ticker], recent_stats_all[ticker])
        render_html(ticker, claude_result, sign_results[ticker], recent_stats_all[ticker], chart_b64)
    except Exception as e:
        print(f"  エラー: {e}")
        import traceback
        traceback.print_exc()